<a href="https://colab.research.google.com/github/henriquebussi/teste_chunks/blob/main/teste_chunks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### *Setup*

**Bibliotecas:**

- **os** - para interagir com o sistema operacional, usa principalmente no Tesseract
- **torch** - para criar, treinar, e rodar modelos de deep learning, nesse código "só" usa para liberar espaço da GPU depois do uso
- **faiss** - para realizar busca de similaridade entre vetores
- **numpy**
- **json** -  para utilizar e fazer arquivos json
- **statistics** - para utilizar algumas formulas matematicas, como a mediana, média e desvio padrão
- **shutil**
- **re**
- **networkx** - para criação e manipulação de grafos, nesse código foi usado no "GraphRAG"
- **community/python-louvain** -
- **pathlip**
- **sentence_transformers**
- **docling**
- **langchain_text_splitters**
- **google.colab** - integração com o ambiente google, usa para utilizar o drive do google

In [ ]:
!pip install -q docling pytesseract sentence-transformers faiss-cpu spacy networkx python-louvain langchain-text-splitters
!sudo apt-get install -y -qq tesseract-ocr tesseract-ocr-por tesseract-ocr-eng

import os
import torch
import faiss
import numpy as np
import json
import statistics
import shutil
import re
import networkx as nx
from community import community_louvain
from pathlib import Path
from sentence_transformers import SentenceTransformer
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TesseractCliOcrOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import ImageRefMode, PictureItem
from langchain_text_splitters import RecursiveCharacterTextSplitter

from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# **Config Inicial**

In [ ]:
# ============================================================
# CONFIGURAÇÃO GLOBAL
# ============================================================
BASE_DIR     = Path("/content/drive/MyDrive/SB100")
OVERLAP_LIST = [128]
CHUNK_LIST   = [800, 1024]

AI_Model = "Qwen/Qwen3-Embedding-0.6B"

# ============================================================
# LISTA DE PDFs PARA TESTAR
# ============================================================
pdfs = [
    # --- Boletim 100 ---
    {
        "nome": "Boletim 100",
        "caminho": "/content/drive/MyDrive/SB100/data/Citrus/Boletim 100_15.07.2022_FINAL - Fernanda Bochi Dos Santos.pdf",
        "language": ["por"],
        "structural": True,
        "recursive": True,
        "perguntas": [
            "De acordo com o Boletim 100, qual é o método de análise de solo utilizado como ponto central para o diagnóstico da disponibilidade de nutrientes em solos tropicais no Estado de São Paulo (exceto para o nitrogênio)?",
            "Para a cultura da alcachofra e do tomate, quais são os valores recomendados de saturação por bases (V%) e o teor mínimo de magnésio (Mg) a serem atingidos através da calagem?",
            "Por que os teores de nutrientes como Cálcio (Ca) e Boro (B) tendem a aumentar nas folhas mais velhas, ao contrário do Nitrogênio (N) e do Potássio (K)?",
            "Explique por que, em solos tropicais ácidos, os baixos teores de Cálcio (Ca) e Magnésio (Mg) estão frequentemente associados a elevados teores de Alumínio (Al³⁺) e quais os impactos disso para as raízes.",
            "Na cultura do fumo, por que o uso de Cloreto de Potássio (KCl) deve ser evitado nas duas últimas aplicações de cobertura, e qual o critério para suspender a adubação nitrogenada?"
        ]
    },
    {
        "nome": "Arantes(2020)-Microbiome",
        "caminho": "/content/drive/MyDrive/SB100/data/Citrus/Application of biostimulant consortium_bioinsumos.pdf",
        "language": ["eng"],
        "structural": True,
        "recursive": True,
        "perguntas": [
            "What are the main components of the biostimulant consortium used in this study?",
            "At what stage after planting was the Sucrosin applied via foliar spray?",
            "How did the application of the biostimulant consortium affect the height and stem diameter of sugarcane plants compared to the control?",
            "What environmental condition during the experiment negatively impacted the growth of sugarcane plants between 4 to 6 months after planting?",
            "Explain how humic acid and mycorrhizal fungi contribute to nutrient availability and plant stress tolerance according to the study.",
            "Discuss the potential mechanisms by which the biostimulant consortium maintains sugarcane vegetative growth under drought stress conditions."
        ]
    },
    # --- Alva(2005)-NUE ---
    {
        "nome": "Alva(2005)-NUE",
        "caminho": "/content/drive/MyDrive/SB100/data/Citrus/Alva(2005)-NUE - Fernanda Bochi.pdf",
        "language": ["eng"],
        "structural": True,
        "recursive": True,
        "perguntas": [
            "What is the main objective of this study regarding nitrogen management?",
            "Why is nitrate-N leaching considered both an economic loss and an environmental concern?",
            "How does water management influence nitrogen uptake efficiency and its transformation in the soil?",
            "List three factors mentioned that affect the biogeochemical transformations of nitrogen in the soil.",
            "Discuss the holistic approach suggested by the authors to minimize non-point source nitrate pollution in groundwater.",
            "Analyze how the mineralization process recycles plant and animal litter within the nitrogen cycle according to the text."
        ]
    },

    # --- Azevedo(2020)-Frontiers ---
    {
        "nome": "Azevedo(2020)-Frontiers",
        "caminho": "/content/drive/MyDrive/SB100/data/Citrus/Azevedo(2020)-Frontiers - Fernanda Bochi Dos Santos.pdf",
        "language": ["eng"],
        "structural": True,
        "recursive": True,
        "perguntas": [
            "What was the specific dwarfing rootstock used for the Tahiti acid lime in this high-density planting study?",
            "Which intercrop species was maintained as mulch in the no-tillage systems evaluated?",
            "Compare the effect of the no-tillage (NT) system and conventional tillage (CT) on potassium (K) concentrations in the leaves.",
            "How did the maintenance of Urochloa ruziziensis mulch influence undesirable weed populations?",
            "Evaluate the benefits of high-density planting (1,157 trees/ha) combined with no-tillage for sustainable citrus production.",
            "Discuss how soil physical and chemical characteristics were influenced by the different tillage systems over the 5-year study period."
        ]
    },
]

## Tesseract

In [ ]:
# ============================================================
# CONFIGURAÇÃO DO TESSERACT
# ============================================================
candidates = [
    "/usr/share/tesseract-ocr/5/tessdata",
    "/usr/share/tesseract-ocr/4.00/tessdata",
    "/usr/share/tesseract-ocr/tessdata",
    "/usr/share/tessdata",
]

tessdata = next((p for p in candidates if os.path.isdir(p)), None)
if tessdata is None:
    tessdata = "/usr/share/tesseract-ocr/5/tessdata"
    os.makedirs(tessdata, exist_ok=True)

os.environ["TESSDATA_PREFIX"] = tessdata
print(f"\nTESSDATA_PREFIX → {tessdata}")

needed  = ["osd.traineddata", "por.traineddata", "eng.traineddata"]
missing = [f for f in needed if not os.path.isfile(os.path.join(tessdata, f))]

if missing:
    print(f"⚠️  Instalando arquivos faltando: {missing}")
    os.system("apt-get install -y -qq tesseract-ocr-por tesseract-ocr-eng 2>&1")
    tessdata = next((p for p in candidates if os.path.isdir(p)), tessdata)
    os.environ["TESSDATA_PREFIX"] = tessdata
    print("✅ Tesseract configurado!")
else:
    print("✅ Tesseract OK!")


TESSDATA_PREFIX → /usr/share/tesseract-ocr/4.00/tessdata
✅ Tesseract OK!


In [ ]:
# ============================================================
# FUNÇÕES
# ============================================================
def clean_text(text):
    text = text.replace("glyph<c=3,font=/CIDFont+F5>", " ")
    text = text.replace("glyph<c=3,font=/CIDFont+F8>", " ")
    text = text.replace("&gt;", "").replace("&lt;", "")
    return text


def pdf_to_markdown(file_path, ocr_lang, md_dir, data_dir):
    base_stem = Path(file_path).stem
    md_path   = data_dir / f"{base_stem}.md"
    if md_path.exists():
        print(f"ℹ️  Markdown já existe, reutilizando!")
        return str(md_path)
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = True
    pipeline_options.do_table_structure = True
    pipeline_options.table_structure_options.do_cell_matching = True
    pipeline_options.ocr_options = TesseractCliOcrOptions(lang=ocr_lang, force_full_page_ocr=True)
    pipeline_options.generate_picture_images = True
    converter = DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
    )
    print(f"🔄 Convertendo PDF...")
    print(f"   Tamanho: {os.path.getsize(file_path)/(1024*1024):.2f} MB")
    result = converter.convert(file_path).document
    for text in getattr(result, "texts", []):
        text.orig = clean_text(getattr(text, "orig", ""))
    for table in getattr(result, "tables", []):
        for cell in getattr(table.data, "table_cells", []):
            cell.text = clean_text(getattr(cell, "text", ""))
    picture_counter = 0
    for element, _ in result.iterate_items():
        if isinstance(element, PictureItem):
            picture_counter += 1
            img_path = md_dir / f"{base_stem}-picture-{picture_counter}.png"
            with img_path.open("wb") as fp:
                element.get_image(result).save(fp, "PNG")
    print(f"   {picture_counter} imagens salvas")
    result_md = result.export_to_markdown()
    with md_path.open("w", encoding="utf-8") as f:
        f.write(result_md)
    print(f"✅ Markdown salvo em: {md_path}")
    return str(md_path)

# ============================================================
# UTILITÁRIO: normaliza ocr_lang para string em qualquer ponto
# ============================================================
def _normalizar_lang(lang) -> str:
    """Aceita str ou list[str] e sempre devolve uma string simples."""
    if isinstance(lang, list):
        lang = lang[0] if lang else "por"
    return lang.strip()


## Fazer Chunks

In [ ]:
# ============================================================
# FLAT CHUNKER
# ============================================================
def fazer_chunks(md_path, chunk_size, overlap=0):
    with open(md_path, 'r', encoding='utf-8') as f:
        content = f.readlines()

    content = [p.strip() for p in content]
    content = [p for p in content if p.replace('-', '').replace('|', '').replace(' ', '').strip()]

    chunks        = []
    current_chunk = ""

    for i, paragraph in enumerate(content):
        if i == 0:
            current_chunk += paragraph
            continue

        if paragraph.startswith("| ") and paragraph.endswith(" |"):
            current_chunk += "\n" + paragraph
            continue

        if len(current_chunk) + len(paragraph) + 1 > chunk_size:
            if current_chunk:
                chunks.append(current_chunk)
                if overlap > 0:
                    overlap_text = current_chunk[-overlap:]
                    space_idx = overlap_text.find(" ")
                    if space_idx != -1:
                        overlap_text = overlap_text[space_idx + 1:]
                    current_chunk = overlap_text + "\n" + paragraph
                else:
                    current_chunk = paragraph
        else:
            current_chunk += "\n" + paragraph

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

# ============================================================
# RECURSIVE CHUNKER — sem alterações, já estava correto
# ============================================================
def fazer_chunks_recursivo(md_path: str, chunk_size: int, overlap: int) -> list[str]:
    """
    Divide o conteúdo markdown em chunks sobrepostos usando o método
    recursivo do LangChain (ResursiveCharacterTextSplitter).
    """
    with open(md_path, "r", encoding="utf-8") as f:
        content = f.read()

    divisor = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = divisor.split_text(content)
    print(f"✅ Recursive: {len(chunks)} chunks gerados via RecursiveCharacterTextSplitter")
    return chunks


# ============================================================
# GRAPHRAG HELPERS
# ============================================================

_SPACY_MODELS: dict = {}

def _get_spacy(lang):
    lang = _normalizar_lang(lang)
    if lang not in _SPACY_MODELS:
        model_name = "pt_core_news_sm" if lang.startswith("por") else "en_core_web_sm"
        try:
            import spacy
            _SPACY_MODELS[lang] = spacy.load(model_name)
        except OSError:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "spacy", "download", model_name], check=True)
            import spacy
            _SPACY_MODELS[lang] = spacy.load(model_name)
        print(f"✅ spaCy model '{model_name}' carregado para lang='{lang}'")
    return _SPACY_MODELS[lang]


def _split_into_sentences(text: str) -> list[str]:
    lines = text.splitlines()
    sentences: list[str] = []
    buffer = ""

    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.startswith("| ") and stripped.endswith(" |"):
            if buffer:
                sentences.append(buffer.strip())
                buffer = ""
            sentences.append(stripped)
            continue
        buffer += " " + stripped

    if buffer.strip():
        sentences.append(buffer.strip())

    result: list[str] = []
    for s in sentences:
        if s.startswith("| "):
            result.append(s)
        else:
            parts = re.split(r"(?<=[.!?])\s+", s)
            result.extend(p.strip() for p in parts if p.strip())
    return result


def _extract_entities(sentences: list[str], nlp) -> list[list[str]]:
    entity_sets: list[list[str]] = []
    for sent in sentences:
        doc = nlp(sent[:1_000])
        ents = [ent.text.lower().strip() for ent in doc.ents if len(ent.text.strip()) > 2]
        if not ents:
            ents = [
                chunk.root.lemma_.lower()
                for chunk in doc.noun_chunks
                if len(chunk.root.lemma_) > 2
            ]
        entity_sets.append(list(set(ents)))
    return entity_sets


def _build_cooccurrence_graph(
    sentences: list[str],
    entity_sets: list[list[str]],
    window: int = 3,
) -> nx.Graph:
    G = nx.Graph()
    for i, ents in enumerate(entity_sets):
        window_ents: list[str] = []
        for j in range(max(0, i - window), min(len(entity_sets), i + window + 1)):
            window_ents.extend(entity_sets[j])
        for a in ents:
            G.add_node(a, sentence_indices=[])
            G.nodes[a].setdefault("sentence_indices", []).append(i)
            for b in window_ents:
                if a != b:
                    if G.has_edge(a, b):
                        G[a][b]["weight"] += 1
                    else:
                        G.add_edge(a, b, weight=1)
    return G


def _assign_community_to_sentences(
    sentences: list[str],
    entity_sets: list[list[str]],
    partition: dict,
) -> list[int]:
    labels: list[int] = []
    for ents in entity_sets:
        if not ents:
            labels.append(-1)
            continue
        votes: dict[int, int] = {}
        for e in ents:
            cid = partition.get(e, -1)
            if cid != -1:
                votes[cid] = votes.get(cid, 0) + 1
        if not votes:
            labels.append(-1)
        else:
            labels.append(max(votes, key=votes.get))
    return labels


def _merge_orphan_sentences(
    sentences: list[str],
    labels: list[int],
) -> tuple[list[str], list[int]]:
    merged_sentences, merged_labels = list(sentences), list(labels)
    for i, lbl in enumerate(merged_labels):
        if lbl == -1:
            for delta in range(1, len(merged_labels)):
                left  = i - delta
                right = i + delta
                if left >= 0 and merged_labels[left] != -1:
                    merged_labels[i] = merged_labels[left]
                    break
                if right < len(merged_labels) and merged_labels[right] != -1:
                    merged_labels[i] = merged_labels[right]
                    break
            else:
                merged_labels[i] = 0
    return merged_sentences, merged_labels


def _communities_to_chunks(
    sentences: list[str],
    labels: list[int],
    chunk_size: int,
    overlap: int,
) -> list[str]:
    from itertools import groupby
    groups: list[tuple[int, list[str]]] = []
    for cid, group in groupby(zip(labels, sentences), key=lambda x: x):
        groups.append((cid, [s for _, s in group]))

    chunks: list[str] = []
    for cid, group_sentences in groups:
        current = ""
        for sent in group_sentences:
            if len(current) + len(sent) + 1 > chunk_size and current:
                chunks.append(current.strip())
                if overlap > 0:
                    tail = current[-overlap:]
                    sp   = tail.find(" ")
                    current = (tail[sp + 1:] if sp != -1 else tail) + " " + sent
                else:
                    current = sent
            else:
                current = (current + " " + sent).strip()
        if current.strip():
            chunks.append(current.strip())
    return [c for c in chunks if c.strip()]


def fazer_chunks_graphrag(
    md_path: str,
    chunk_size: int,
    overlap: int = 0,
    ocr_lang = "por",
    cooccurrence_window: int = 3,
) -> list[str]:
    ocr_lang = _normalizar_lang(ocr_lang)
    if isinstance(ocr_lang, list):
        ocr_lang = ocr_lang

    with open(md_path, "r", encoding="utf-8") as f:
        content = f.read()

    nlp       = _get_spacy(ocr_lang)
    sentences = _split_into_sentences(content)

    if not sentences:
        print("⚠️  GraphRAG: nenhuma sentença encontrada, fallback para fazer_chunks")
        return fazer_chunks(md_path, chunk_size, overlap)

    print(f"   GraphRAG: {len(sentences)} sentenças extraídas")

    entity_sets = _extract_entities(sentences, nlp)
    print(f"   GraphRAG: extração de entidades concluída")

    G = _build_cooccurrence_graph(sentences, entity_sets, window=cooccurrence_window)
    print(f"   GraphRAG: grafo com {G.number_of_nodes()} nós, {G.number_of_edges()} arestas")

    if G.number_of_nodes() < 2:
        print("⚠️  GraphRAG: grafo muito esparso, fallback para fazer_chunks")
        return fazer_chunks(md_path, chunk_size, overlap)

    partition     = community_louvain.best_partition(G, weight="weight", random_state=42)
    n_communities = len(set(partition.values()))
    print(f"   GraphRAG: {n_communities} comunidades detectadas (Louvain)")

    if n_communities < 3:
        print("⚠️  GraphRAG: poucas comunidades, fallback para fazer_chunks")
        return fazer_chunks(md_path, chunk_size, overlap)

    labels            = _assign_community_to_sentences(sentences, entity_sets, partition)
    sentences, labels = _merge_orphan_sentences(sentences, labels)
    chunks            = _communities_to_chunks(sentences, labels, chunk_size, overlap)

    print(f"✅ GraphRAG: {len(chunks)} chunks gerados via comunidades de conhecimento")
    return chunks


# ============================================================
# SELECTOR
# ============================================================
def selecionar_chunks(
    md_path: str,
    chunk_size: int,
    overlap: int,
    ocr_lang,
    structural: bool = False,
    recursive: bool = False,
) -> list[str]:

    ocr_lang = _normalizar_lang(ocr_lang)
    if structural:
        print(f"📐 Modo: STRUCTURAL (GraphRAG) | chunk_size={chunk_size} | overlap={overlap}")
        return fazer_chunks_graphrag(md_path, chunk_size, overlap, ocr_lang)
    elif recursive:
        print(f"📐 Modo: RECURSIVE (LangChain) | chunk_size={chunk_size} | overlap={overlap}")
        return fazer_chunks_recursivo(md_path, chunk_size, overlap)
    else:
        print(f"📐 Modo: SEMANTIC/FLAT (fazer_chunks)   | chunk_size={chunk_size} | overlap={overlap}")
        return fazer_chunks(md_path, chunk_size, overlap)


## Avaliação FAISS

In [ ]:
# ============================================================
# AVALIAÇÃO, FAISS, BUSCA
# ============================================================
def avaliar_tamanho_chunks(chunks, label):
    tamanhos  = [len(c) for c in chunks]
    ideal     = sum(1 for t in tamanhos if 300 <= t <= 1024)
    score_tam = (ideal / len(chunks)) * 100
    print(f"\n{'='*55}")
    print(f"📏 AVALIAÇÃO DE TAMANHO — {label}")
    print(f"{'='*55}")
    print(f"   Total chunks  : {len(chunks)}")
    print(f"   Menor         : {min(tamanhos)} chars")
    print(f"   Maior         : {max(tamanhos)} chars")
    print(f"   Média         : {statistics.mean(tamanhos):.0f} chars")
    print(f"   Mediana       : {statistics.median(tamanhos):.0f} chars")
    print(f"   Desvio padrão : {statistics.stdev(tamanhos):.0f} chars")
    print(f"\n   Distribuição:")
    print(f"   < 200  : {sum(1 for t in tamanhos if t < 200):>4} ⚠️")
    print(f"   200-400: {sum(1 for t in tamanhos if 200 <= t < 400):>4}")
    print(f"   400-600: {sum(1 for t in tamanhos if 400 <= t < 600):>4} ✅")
    print(f"   600-800: {sum(1 for t in tamanhos if 600 <= t < 800):>4} ✅")
    print(f"   800-1024:{sum(1 for t in tamanhos if 800 <= t < 1024):>4} ✅")
    print(f"   > 1024 : {sum(1 for t in tamanhos if t > 1024):>4} ⚠️")
    print(f"\n   ⭐ Score tamanho: {score_tam:.1f}% ({ideal}/{len(chunks)} no ideal)")
    return score_tam


def criar_indice_faiss(chunks: list[str]):
    print(f"🔄 Gerando embeddings para {len(chunks)} chunks...")
    torch.cuda.empty_cache()
    embeddings = model.encode(
        chunks,
        show_progress_bar=True,
        batch_size=4,
        normalize_embeddings=True,
    )
    embeddings = np.array(embeddings).astype("float32")
    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    print(f"✅ Índice FAISS criado com {index.ntotal} vetores (dim={dim})!")
    return index


def buscar_faiss(pergunta: str, index, chunks: list[str]) -> dict:
    vetor = model.encode([pergunta], normalize_embeddings=True)
    vetor = np.array(vetor).astype("float32")
    faiss.normalize_L2(vetor)
    scores, indices = index.search(vetor, 1)
    idx = indices[0][0]
    return {
        "score": float(scores[0][0]),
        "chunk": chunks[idx],
        "chars": len(chunks[idx]),
    }

## Loop Principal e Embedding

In [ ]:
# ============================================================
# EMBEDDING
# ============================================================
print(f"\n🔄 Carregando modelo de embedding {AI_Model}...")
model = SentenceTransformer(AI_Model, device='cuda')
print("✅ Modelo carregado!")


# ============================================================
# LOOP PRINCIPAL — agrupa por (nome, caminho) e itera modos
# ============================================================
BASE_DIR = Path("/content/drive/MyDrive/SB100")

todos_resultados_global = {}

# Deduplica: um entry por PDF único (mesmo nome+caminho)
pdfs_unicos = {}
for pdf_cfg in pdfs:
    chave = (pdf_cfg["nome"], pdf_cfg["caminho"])
    if chave not in pdfs_unicos:
        pdfs_unicos[chave] = pdf_cfg  # guarda o primeiro só pra metadados

for (nome, caminho), pdf_cfg in pdfs_unicos.items():

    OCR_LANG  = pdf_cfg["language"]
    perguntas = pdf_cfg["perguntas"]

    slug     = nome.replace(" ", "_").replace("/", "-")
    MD_DIR   = BASE_DIR / "testes" / slug / "md_images"
    DATA_DIR = BASE_DIR / "testes" / slug / "data_md"
    MD_DIR.mkdir(parents=True, exist_ok=True)
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    original_pdf_path = caminho
    if original_pdf_path.startswith("/content/drive/MyDrive/SB100"):
        PDF_TESTE = original_pdf_path.replace("/content/drive/MyDrive/SB100", str(BASE_DIR))
    else:
        PDF_TESTE = original_pdf_path

    print(f"\n{'█'*60}")
    print(f"█ PDF: {nome}")
    print(f"█ Idioma: {OCR_LANG} | Perguntas: {len(perguntas)}")
    print(f"{'█'*60}")

    md_path = pdf_to_markdown(PDF_TESTE, OCR_LANG, MD_DIR, DATA_DIR)

    todos_resultados = []

    # ← agora itera os 3 modos para cada PDF
    MODOS = [
        {"modo_str": "flat",       "structural": False, "recursive": False},
        {"modo_str": "recursive",  "structural": False, "recursive": True},
        {"modo_str": "structural", "structural": True,  "recursive": False},
    ]

    for modo_cfg in MODOS:
        modo_str   = modo_cfg["modo_str"]
        STRUCTURAL = modo_cfg["structural"]
        RECURSIVE  = modo_cfg["recursive"]

        modo_display = "STRUCTURAL (graph)" if STRUCTURAL else "RECURSIVE (LangChain)" if RECURSIVE else "FLAT (fazer_chunks)"
        print(f"\n{'─'*60}")
        print(f"  MODO: {modo_display}")
        print(f"{'─'*60}")

        for chunk_size in CHUNK_LIST:
            for overlap in OVERLAP_LIST:

                label = f"chunk_{chunk_size}_overlap_{overlap}_{modo_str}"

                print(f"\n{'#'*60}")
                print(f"# {nome} | {modo_str} | chunk={chunk_size} | overlap={overlap}")
                print(f"{'#'*60}")

                chunks    = selecionar_chunks(md_path, chunk_size, overlap, OCR_LANG, structural=STRUCTURAL, recursive=RECURSIVE)
                score_tam = avaliar_tamanho_chunks(chunks, label)
                index     = criar_indice_faiss(chunks)

                print(f"\n{'='*60}")
                print(f"🔍 RESULTADOS — {nome} | {modo_str} | chunk={chunk_size} | overlap={overlap}")
                print(f"{'='*60}")

                scores = []
                for pergunta in perguntas:
                    r         = buscar_faiss(pergunta, index, chunks)
                    relevante = "✅" if r['score'] > 0.55 else "⚠️" if r['score'] > 0.45 else "❌"
                    scores.append(r['score'])
                    print(f"\n❓ {pergunta}")
                    print(f"   Score : {r['score']:.4f} {relevante} | Chars: {r['chars']}")
                    print(f"   Texto : {r['chunk'][:200]}...")
                    print("-"*60)

                media_score = sum(scores) / len(scores)
                print(f"\n📈 MÉDIA  : {media_score:.4f}")
                print(f"   Tamanho: {score_tam:.1f}%")
                print(f"   ✅ bons : {sum(1 for s in scores if s > 0.55)}/{len(scores)}")
                print(f"   ⚠️ ok   : {sum(1 for s in scores if 0.45 <= s <= 0.55)}/{len(scores)}")
                print(f"   ❌ ruins: {sum(1 for s in scores if s < 0.45)}/{len(scores)}")

                resultado = {
                    "chunk_size":   chunk_size,
                    "overlap":      overlap,
                    "label":        label,
                    "modo":         modo_str,
                    "total_chunks": len(chunks),
                    "menor_chunk":  min(len(c) for c in chunks),
                    "maior_chunk":  max(len(c) for c in chunks),
                    "media_chunk":  sum(len(c) for c in chunks) // len(chunks),
                    "score_tamanho": score_tam,
                    "scores":       scores,
                    "media_score":  media_score,
                    "md_nome":      Path(md_path).stem
                }
                todos_resultados.append(resultado)

    # salva JSON por PDF
    output_path = f"/content/resultado_{slug}.json"
    with open(output_path, "w") as f:
        json.dump({
            "pdf":        nome,
            "modelo":     AI_Model,
            "idioma":     OCR_LANG,
            "overlap_list": OVERLAP_LIST,
            "chunk_list": CHUNK_LIST,
            "resultados": todos_resultados          # contém os 3 modos
        }, f, ensure_ascii=False, indent=2)
    print(f"\n✅ Salvo em: {output_path}")

    print(f"\n🏆 RANKING — {nome}:")
    ranking = sorted(todos_resultados, key=lambda x: x['media_score'], reverse=True)
    for i, r in enumerate(ranking, 1):
        emoji = "✅" if r['media_score'] > 0.55 else "⚠️" if r['media_score'] > 0.45 else "❌"
        print(f"   {i}º {r['label']} → score={r['media_score']:.4f} {emoji} | tam={r['score_tamanho']:.1f}%")

    todos_resultados_global[nome] = todos_resultados   # ← written once per PDF

# salva consolidado
with open("/content/resultado_consolidado.json", "w") as f:
    json.dump(todos_resultados_global, f, ensure_ascii=False, indent=2)

print(f"\n{'='*60}")
print(f"✅ TODOS OS PDFs PROCESSADOS!")
print(f"   PDFs testados: {list(pdfs_unicos.keys())}")
print(f"   Consolidado em: /content/resultado_consolidado.json")


🔄 Carregando modelo de embedding Qwen/Qwen3-Embedding-0.6B...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

✅ Modelo carregado!

████████████████████████████████████████████████████████████
█ PDF: Boletim 100
█ Idioma: ['por'] | Perguntas: 5
████████████████████████████████████████████████████████████
ℹ️  Markdown já existe, reutilizando!

────────────────────────────────────────────────────────────
  MODO: FLAT (fazer_chunks)
────────────────────────────────────────────────────────────

############################################################
# Boletim 100 | flat | chunk=800 | overlap=128
############################################################
📐 Modo: SEMANTIC/FLAT (fazer_chunks)   | chunk_size=800 | overlap=128

📏 AVALIAÇÃO DE TAMANHO — chunk_800_overlap_128_flat
   Total chunks  : 1453
   Menor         : 143 chars
   Maior         : 14054 chars
   Média         : 942 chars
   Mediana       : 724 chars
   Desvio padrão : 979 chars

   Distribuição:
   < 200  :   12 ⚠️
   200-400:   80
   400-600:  296 ✅
   600-800:  638 ✅
   800-1024: 146 ✅
   > 1024 :  281 ⚠️

   ⭐ Score tamanho:

Batches:   0%|          | 0/364 [00:00<?, ?it/s]

✅ Índice FAISS criado com 1453 vetores (dim=1024)!

🔍 RESULTADOS — Boletim 100 | flat | chunk=800 | overlap=128

❓ De acordo com o Boletim 100, qual é o método de análise de solo utilizado como ponto central para o diagnóstico da disponibilidade de nutrientes em solos tropicais no Estado de São Paulo (exceto para o nitrogênio)?
   Score : 0.7036 ✅ | Chars: 424
   Texto : por meio da saturação por bases do solo (V). Também, resultados da análise química de amostras do subsolo são interpretados.
A presente edição mantém as interpretações para respostas a N da versão ant...
------------------------------------------------------------

❓ Para a cultura da alcachofra e do tomate, quais são os valores recomendados de saturação por bases (V%) e o teor mínimo de magnésio (Mg) a serem atingidos através da calagem?
   Score : 0.6968 ✅ | Chars: 453
   Texto : calcário para elevar a saturação por bases para os valores de 80% e garantir o mínimo de 9 mmol dm? de magnésio trocável;
Pasto consorciado

...

✅ Índice FAISS criado com 84 vetores (dim=1024)!

🔍 RESULTADOS — Azevedo(2020)-Frontiers | recursive | chunk=1024 | overlap=128

❓ What was the specific dwarfing rootstock used for the Tahiti acid lime in this high-density planting study?
   Score : 0.7567 ✅ | Chars: 849
   Texto : ## Vegetative and Productive Development

In the fifth year of evaluation, the canopy volume of the Tahiti acid lime was still small, due to the use of the Flying Dragon trifoliate orange, a dwarfing ...
------------------------------------------------------------

❓ Which intercrop species was maintained as mulch in the no-tillage systems evaluated?
   Score : 0.6993 ✅ | Chars: 684
   Texto : ## CONCLUSION

The reported no-tillage (NT) system, including the correct management of intercropping Urochloa ruziziensis with the use of an ecological mower, and glyphosate in rows, has improved the...
------------------------------------------------------------

❓ Compare the effect of the no-tillage (NT) system and

Batches:   0%|          | 0/236 [00:00<?, ?it/s]

✅ Índice FAISS criado com 944 vetores (dim=1024)!

🔍 RESULTADOS — Azevedo(2020)-Frontiers | structural | chunk=800 | overlap=128

❓ What was the specific dwarfing rootstock used for the Tahiti acid lime in this high-density planting study?
   Score : 0.8439 ✅ | Chars: 205
   Texto : Thus, the present research aimed to evaluate different planting systems for Tahiti acid lime grafted onto Flying Dragon trifoliate orange, a dwarfing rootstock, at high planting density (1,157 trees h...
------------------------------------------------------------

❓ Which intercrop species was maintained as mulch in the no-tillage systems evaluated?
   Score : 0.7793 ✅ | Chars: 268
   Texto : The study was set up in four tillage systems, using Urochloa ruziziensis as an intercrop species in the orchard, and conducted for 5 years: no-tillage (NT), no-tillage and no-herbicide (NT-NH), minimu...
------------------------------------------------------------

❓ Compare the effect of the no-tillage (NT) system an

Batches:   0%|          | 0/236 [00:00<?, ?it/s]

✅ Índice FAISS criado com 944 vetores (dim=1024)!

🔍 RESULTADOS — Azevedo(2020)-Frontiers | structural | chunk=1024 | overlap=128

❓ What was the specific dwarfing rootstock used for the Tahiti acid lime in this high-density planting study?
   Score : 0.8439 ✅ | Chars: 205
   Texto : Thus, the present research aimed to evaluate different planting systems for Tahiti acid lime grafted onto Flying Dragon trifoliate orange, a dwarfing rootstock, at high planting density (1,157 trees h...
------------------------------------------------------------

❓ Which intercrop species was maintained as mulch in the no-tillage systems evaluated?
   Score : 0.7793 ✅ | Chars: 268
   Texto : The study was set up in four tillage systems, using Urochloa ruziziensis as an intercrop species in the orchard, and conducted for 5 years: no-tillage (NT), no-tillage and no-herbicide (NT-NH), minimu...
------------------------------------------------------------

❓ Compare the effect of the no-tillage (NT) system a

## Ranking

In [ ]:

# ============================================================
# RANKING CONSOLIDADO FINAL
# ============================================================
print(f"\n{'█'*60}")
print(f"█ RANKING CONSOLIDADO — TODOS OS PDFs")
print(f"{'█'*60}")

todos_flat = []
for nome_pdf, resultados in todos_resultados_global.items():
    for r in resultados:
        todos_flat.append({**r, "pdf": nome_pdf})

ranking_global = sorted(todos_flat, key=lambda x: x['media_score'], reverse=True)

print(f"\n{'Pos':>4} | {'PDF':<30} | {'Label':<45} | {'Score':>7} | {'Tam%':>6} | {'Chunks':>6} | {'Modo':<10}")
print("-" * 120)
for i, r in enumerate(ranking_global, 1):
    relevantes = sum(1 for s in r['scores'] if s > 0.55)
    ok         = sum(1 for s in r['scores'] if 0.45 <= s <= 0.55)
    ruins      = sum(1 for s in r['scores'] if s < 0.45)
    emoji      = "✅" if r['media_score'] > 0.55 else "⚠️" if r['media_score'] > 0.45 else "❌"
    print(
        f"{i:>4} | {r['pdf']:<30} | {r['label']:<45} | "
        f"{r['media_score']:.4f} {emoji} | {r['score_tamanho']:>5.1f}% | "
        f"{r['total_chunks']:>6} | {r['modo']:<10} | "
        f"✅{relevantes} ⚠️{ok} ❌{ruins}"
    )

print(f"\n🥇 MELHOR CONFIGURAÇÃO GLOBAL:")
best = ranking_global[0]
print(f"  PDF    : {best['pdf']}")
print(f"  Label  : {best['label']}")
print(f"  Score  : {best['media_score']:.4f}")
print(f"  Tamanho: {best['score_tamanho']:.1f}%")
print(f"  Chunks : {best['total_chunks']} (min={best['menor_chunk']} / med={best['media_chunk']} / max={best['maior_chunk']} chars)")
print(f"  Modo   : {best['modo']}")


████████████████████████████████████████████████████████████
█ RANKING CONSOLIDADO — TODOS OS PDFs
████████████████████████████████████████████████████████████

 Pos | PDF                            | Label                                         |   Score |   Tam% | Chunks | Modo      
------------------------------------------------------------------------------------------------------------------------
   1 | Arantes(2020)-Microbiome       | chunk_800_overlap_128_structural              | 0.8280 ✅ |   1.8% |    164 | structural | ✅6 ⚠️0 ❌0
   2 | Arantes(2020)-Microbiome       | chunk_1024_overlap_128_structural             | 0.8280 ✅ |   1.8% |    164 | structural | ✅6 ⚠️0 ❌0
   3 | Alva(2005)-NUE                 | chunk_800_overlap_128_structural              | 0.7962 ✅ |   0.6% |   1896 | structural | ✅6 ⚠️0 ❌0
   4 | Alva(2005)-NUE                 | chunk_1024_overlap_128_structural             | 0.7962 ✅ |   0.6% |   1896 | structural | ✅6 ⚠️0 ❌0
   5 | Arantes(2020)-Microbiom